# ASAP8 preliminary voltage characterization

This notebook is a meeting-oriented first pass at raw ASAP8 voltage data. It keeps the original reference-image and full-session trace views, then adds per-ROI preprocessing, conservative event/spike detection, spike-triggered averages, plateau/compound-event metrics, and between-ROI comparisons.

The spike/event detector here is intentionally exploratory rather than a final sorter. The key output to trust first is the overlay plot: detection thresholds should be tuned until the red markers match what you would manually call spikes or compound depolarizing events.

In [ ]:

# Core imports
import os
import glob
import json
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage, signal

# Use inline figures for robust notebook rendering/saving.

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

try:
    from vip_slap2_analysis.utils.utils import save_figure
except Exception:
    def save_figure(fig, out_base, formats=(".png",), dpi=300):
        """Fallback save_figure implementation when the repo helper is unavailable."""
        out_base = Path(out_base)
        out_base.parent.mkdir(parents=True, exist_ok=True)
        for ext in formats:
            fig.savefig(str(out_base.with_suffix(ext)), dpi=dpi, bbox_inches="tight")

In [ ]:
%matplotlib notebook

## Paths and notebook switches

By default this looks for the uploaded/example files next to the notebook. For a real session, point `SESSION_PATH` to the SLAP2 session folder or set `SUMMARY_PATH` and `TRACE_H5_PATH` directly.

In [ ]:

# ---- Edit these as needed ----
SESSION_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP8\852835\852835_2026-07-29_10-08-37")


summary_hits = sorted(SESSION_PATH.glob("**/dendriticVoltageSummary*.mat"))
trace_hits = sorted(SESSION_PATH.glob("**/dendriticVoltageTraces*.h5"))
if not summary_hits or not trace_hits:
    raise FileNotFoundError("Could not find voltage summary/traces. Update SESSION_PATH, SUMMARY_PATH, or TRACE_H5_PATH.")
SUMMARY_PATH = summary_hits[0]
TRACE_H5_PATH = trace_hits[0]

FIG_DIR = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures"
SAVE_FIGURES = True

print(f"SUMMARY_PATH:  {SUMMARY_PATH.resolve()}")
print(f"TRACE_H5_PATH: {TRACE_H5_PATH.resolve()}")

## Voltage summary and dual-mode trace loader

The loader below preserves native continuous loading when `/traces/continuous/DMD#` exists. For trial-only H5 outputs, it reconstructs a DMD-local session stream by concatenating `/traces/trial_XXXX` in order and retains trial boundaries plus the inferred acquisition timebase from the summary metadata.

In [ ]:
def _as_scalar(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return np.nan
    arr = np.squeeze(arr)
    if arr.size == 1:
        value = arr.reshape(-1)[0]
        return value.item() if isinstance(value, np.generic) else value
    return arr


def _decode_matlab_char(arr):
    """Decode MATLAB char arrays stored as uint16."""
    arr = np.asarray(arr).squeeze()
    if arr.size == 0:
        return ""
    return "".join(chr(int(c)) for c in arr if int(c) != 0)


def _read_ref(f, ref):
    return f[ref]


def read_summary_scalar(summary_path, h5_path):
    with h5py.File(summary_path, "r") as f:
        return _as_scalar(f[h5_path][()])


def read_summary_text(summary_path, h5_path):
    with h5py.File(summary_path, "r") as f:
        return _decode_matlab_char(f[h5_path][()])


def orient_slap2_image_for_display(im, flip_y=True):
    out = np.asarray(im).T
    return np.flipud(out) if flip_y else out


def orient_slap2_masks_for_display(masks, flip_y=True):
    out = np.asarray(masks).transpose(0, 2, 1)
    return out[:, ::-1, :] if flip_y else out


def read_ref_image(summary_path, dmd=1, for_display=False, flip_y=True):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/refIM"][dmd - 1, 0]
        im = np.asarray(_read_ref(f, ref)[()], dtype=np.float32)
    return orient_slap2_image_for_display(im, flip_y=flip_y) if for_display else im


def read_roi_masks(summary_path, dmd=1, for_display=False, flip_y=True):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/masks"][dmd - 1, 0]
        masks = np.asarray(_read_ref(f, ref)[()], dtype=bool)
    return orient_slap2_masks_for_display(masks, flip_y=flip_y) if for_display else masks


def read_mask_image(summary_path, dmd=1, for_display=False, flip_y=True):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/maskImages"][dmd - 1, 0]
        im = np.asarray(_read_ref(f, ref)[()], dtype=np.float32)
    return orient_slap2_image_for_display(im, flip_y=flip_y) if for_display else im


def get_dmd_metadata(summary_path, dmd=1):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/dmd/metadata"][dmd - 1, 0]
        grp = _read_ref(f, ref)
        return {k: _as_scalar(v[()]) for k, v in grp.items()}


def _orient_trace_dataset(ds, expected_n_rois):
    """Return an H5 trace dataset as ROI x time without guessing from channel count."""
    arr = np.asarray(ds[()], dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D trace dataset, got shape {arr.shape}")
    if arr.shape[0] == expected_n_rois:
        return arr
    if arr.shape[1] == expected_n_rois:
        return arr.T
    raise ValueError(
        f"Neither axis matches expected_n_rois={expected_n_rois}; dataset shape={arr.shape}"
    )


def _read_trial_timing(summary_path, n_trials, fs, trial_lengths):
    """Read trial timing and construct both compressed and acquisition-time vectors."""
    starts = None
    ends = None
    with h5py.File(summary_path, "r") as f:
        if "summary/trialTable/trialStartTimeInferred" in f:
            starts = np.asarray(
                f["summary/trialTable/trialStartTimeInferred"][()], dtype=float
            ).reshape(-1)[:n_trials]
        if "summary/trialTable/trialEndTimeFromPC" in f:
            ends = np.asarray(
                f["summary/trialTable/trialEndTimeFromPC"][()], dtype=float
            ).reshape(-1)[:n_trials]

    total = int(np.sum(trial_lengths))
    compressed_time_sec = np.arange(total, dtype=np.float64) / float(fs)
    acquisition_time_sec = np.empty(total, dtype=np.float64)
    trial_id = np.empty(total, dtype=np.int32)
    trial_slices = []

    cursor = 0
    for i, n in enumerate(trial_lengths):
        n = int(n)
        sl = slice(cursor, cursor + n)
        trial_id[sl] = i + 1
        trial_slices.append({"trial": i + 1, "start": cursor, "stop": cursor + n})
        if starts is not None and starts.size == n_trials and np.all(np.isfinite(starts)):
            # MATLAB datenums are in days. Set the first trial start to t=0.
            start_sec = (starts[i] - starts[0]) * 86400.0
            acquisition_time_sec[sl] = start_sec + np.arange(n, dtype=np.float64) / float(fs)
        else:
            acquisition_time_sec[sl] = compressed_time_sec[sl]
        cursor += n

    timing = {
        "compressed_time_sec": compressed_time_sec,
        "acquisition_time_sec": acquisition_time_sec,
        "trial_id": trial_id,
        "trial_slices": trial_slices,
        "trial_start_matlab_datenum": starts,
        "trial_end_matlab_datenum": ends,
    }
    return timing


def load_voltage_session_traces(summary_path, trace_h5_path, fs, mode="auto"):
    """Load DMD traces as ROI x session-time from either H5 storage mode.

    Parameters
    ----------
    mode : {'auto', 'continuous', 'trial'}
        ``auto`` prefers native ``/traces/continuous/DMD#`` datasets. If they are
        absent, it concatenates ``/traces/trial_XXXX`` datasets in trial order.
        ``continuous`` requires native continuous storage. ``trial`` explicitly
        reconstructs a session stream from trial datasets.

    Returns
    -------
    traces : dict[int, ndarray]
        DMD-local arrays shaped ``(n_rois, n_session_samples)``.
    info : dict
        Storage mode, trial boundaries, and continuous/acquisition time vectors.

    Notes
    -----
    Trial concatenation is lossless for the stored samples. ``compressed_time_sec``
    removes any inter-trial gaps; ``acquisition_time_sec`` uses
    ``trialStartTimeInferred`` from the summary and therefore preserves such gaps
    when they exist.
    """
    mode = str(mode).lower()
    if mode not in {"auto", "continuous", "trial"}:
        raise ValueError("mode must be 'auto', 'continuous', or 'trial'")

    with h5py.File(summary_path, "r") as sf:
        n_rois = np.asarray(sf["summary/nAnalysisROIs"][()], dtype=int).reshape(-1)
        if "summary/roiGlobalOffsets" in sf:
            offsets = np.asarray(sf["summary/roiGlobalOffsets"][()], dtype=int).reshape(-1)
        else:
            offsets = np.r_[0, np.cumsum(n_rois[:-1])]

    with h5py.File(trace_h5_path, "r") as tf:
        continuous_available = "traces/continuous" in tf and all(
            f"traces/continuous/DMD{i+1}" in tf for i in range(len(n_rois))
        )
        trial_keys = sorted(
            k for k in tf.get("traces", {}).keys() if k.startswith("trial_")
        )
        trial_available = len(trial_keys) > 0

        selected_mode = mode
        if mode == "auto":
            selected_mode = "continuous" if continuous_available else "trial"

        if selected_mode == "continuous":
            if not continuous_available:
                raise KeyError(
                    "Native continuous traces were requested, but /traces/continuous/DMD# "
                    "is absent. Use mode='auto' or mode='trial' to reconstruct the session stream."
                )
            traces = {
                dmd: _orient_trace_dataset(
                    tf[f"traces/continuous/DMD{dmd}"], int(n_rois[dmd - 1])
                )
                for dmd in range(1, len(n_rois) + 1)
            }
            n_samples = {x.shape[1] for x in traces.values()}
            if len(n_samples) != 1:
                raise ValueError(f"Continuous DMD datasets have different lengths: {n_samples}")
            n = n_samples.pop()
            info = {
                "mode": "continuous",
                "native_continuous": True,
                "continuous_available": True,
                "trial_available": trial_available,
                "compressed_time_sec": np.arange(n, dtype=np.float64) / float(fs),
                "acquisition_time_sec": np.arange(n, dtype=np.float64) / float(fs),
                "trial_id": None,
                "trial_slices": None,
            }
            return traces, info

        if not trial_available:
            raise KeyError("No /traces/trial_XXXX datasets were found.")

        chunks = {dmd: [] for dmd in range(1, len(n_rois) + 1)}
        trial_lengths = []
        n_total_rois = int(np.sum(n_rois))

        for key in trial_keys:
            full = _orient_trace_dataset(tf[f"traces/{key}"], n_total_rois)
            trial_lengths.append(full.shape[1])
            for dmd in chunks:
                i0 = int(offsets[dmd - 1])
                i1 = i0 + int(n_rois[dmd - 1])
                chunks[dmd].append(full[i0:i1, :])

        traces = {dmd: np.concatenate(parts, axis=1) for dmd, parts in chunks.items()}
        timing = _read_trial_timing(summary_path, len(trial_keys), fs, trial_lengths)
        info = {
            "mode": "trial_concatenated",
            "native_continuous": False,
            "continuous_available": continuous_available,
            "trial_available": True,
            "trial_lengths_samples": np.asarray(trial_lengths, dtype=np.int64),
            **timing,
        }
        return traces, info


def describe_trace_h5(trace_h5_path):
    rows = []
    with h5py.File(trace_h5_path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows.append({
                    "dataset": name,
                    "shape": obj.shape,
                    "dtype": str(obj.dtype),
                    "chunks": obj.chunks,
                    "compression": obj.compression,
                })
        f.visititems(visitor)
        attrs = dict(f.attrs)
    return pd.DataFrame(rows), attrs


In [ ]:
# Load core metadata and traces
trace_inventory, trace_attrs = describe_trace_h5(TRACE_H5_PATH)
display(trace_inventory)
print("H5 attrs:")
print(json.dumps({k: str(v) for k, v in trace_attrs.items()}, indent=2))

n_dmds = int(read_summary_scalar(SUMMARY_PATH, "summary/nDMDs"))
n_total_rois = int(read_summary_scalar(SUMMARY_PATH, "summary/nTotalROIs"))
print(f"n_dmds={n_dmds}, n_total_rois={n_total_rois}")

metadata = {dmd: get_dmd_metadata(SUMMARY_PATH, dmd=dmd) for dmd in range(1, n_dmds + 1)}
line_rates = np.asarray([
    np.asarray(metadata[dmd].get("lineRateHz", 10800.0), dtype=float).reshape(-1)[0]
    for dmd in range(1, n_dmds + 1)
])
if not np.allclose(line_rates, line_rates[0], rtol=1e-6, atol=1e-6):
    raise ValueError(f"DMD line rates differ: {line_rates}")
FS = float(line_rates[0])
print(f"Using sample rate FS={FS:.3f} Hz")

# TRACE_MODE options:
#   'auto'       -> native continuous if present, otherwise trial concatenation
#   'continuous' -> require /traces/continuous/DMD#
#   'trial'      -> explicitly concatenate /traces/trial_XXXX datasets
TRACE_MODE = "auto"
traces, TRACE_INFO = load_voltage_session_traces(
    SUMMARY_PATH,
    TRACE_H5_PATH,
    fs=FS,
    mode=TRACE_MODE,
)

# Most downstream cells can use TIME_SEC exactly as they did for native continuous data.
# For trial-only files, acquisition_time_sec preserves inferred trial timing/gaps.
TIME_SEC = TRACE_INFO["acquisition_time_sec"]
COMPRESSED_TIME_SEC = TRACE_INFO["compressed_time_sec"]
TRIAL_ID = TRACE_INFO["trial_id"]
TRIAL_SLICES = TRACE_INFO["trial_slices"]

print(f"Loaded trace mode: {TRACE_INFO['mode']}")
for dmd, X in traces.items():
    print(
        f"DMD{dmd}: {X.shape[0]} ROIs x {X.shape[1]} samples, "
        f"duration={X.shape[1] / FS:.2f} s"
    )

if TRACE_INFO["mode"] == "trial_concatenated":
    lengths = TRACE_INFO["trial_lengths_samples"]
    print(
        f"Reconstructed from {len(lengths)} trials; "
        f"trial durations={lengths.min()/FS:.3f}-{lengths.max()/FS:.3f} s."
    )
    gap = np.diff(TIME_SEC) - 1.0 / FS
    print(
        f"Largest inferred inter-trial timing discontinuity: "
        f"{np.nanmax(np.abs(gap))*1e3:.3f} ms"
    )


## Reference images and ROI masks

The first figure reproduces the core anatomical/targeting view, but also overlays ROI masks and labels to make the subsequent per-ROI traces easier to interpret.

In [ ]:

def overlay_roi_masks(ax, masks, label_prefix="", contour_lw=0.8):
    """Overlay ROI contours and index labels on an image axis."""
    for roi_idx, mask in enumerate(masks):
        if np.any(mask):
            ax.contour(mask.astype(float), levels=[0.5], linewidths=contour_lw)
            yy, xx = np.nonzero(mask)
            ax.text(float(np.mean(xx)), float(np.mean(yy)), f"{label_prefix}{roi_idx}",
                    ha="center", va="center", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.65))

# Reference images are read from the MATLAB/HDF5 summary as (x, y), then
# transposed to (y, x) for a horizontal SLAP2-style display.
fig, axes = plt.subplots(1, n_dmds, figsize=(7.5 * n_dmds, 4.8), sharex=True, sharey=True)
if n_dmds == 1:
    axes = [axes]

ref_images = {}
roi_masks = {}
for ax, dmd in zip(axes, range(1, n_dmds + 1)):
    ref = read_ref_image(SUMMARY_PATH, dmd=dmd, for_display=True)[:,:,0]
    masks = read_roi_masks(SUMMARY_PATH, dmd=dmd, for_display=True)
    ref_images[dmd] = ref
    roi_masks[dmd] = masks
    vmax = np.nanpercentile(ref[ref > -1], 99.5) if np.any(ref > -1) else np.nanmax(ref)
    ax.imshow(ref, cmap="bone", vmin=0, vmax=vmax, origin="upper")
    ax.set_aspect("equal")
    overlay_roi_masks(ax, masks)
    ax.set_title(f"DMD{dmd}: reference image + ROI masks")
    ax.set_xlabel("X (px)")
axes[0].set_ylabel("Y (px)")
fig.tight_layout()

if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"01_reference_images_roi_masks"), formats=(".png", ".pdf"))

### NOTE TO SELF
When selecting ROIS:
- Draw donuts around cells
- If cell bodies are not arranged in columns, use ~10kHz sampling. Else, use 2kHz and ensure >5 cell bodies in one column
- If sampling at >2kHz, decrease laser power from 100% to ~68%

## Full-session traces

These plots are useful for spotting gross differences in expression/brightness, slow drift, high-rate bursting, and obvious artifacts. The traces are vertically offset in raw fluorescence units.

In [ ]:

def plot_stacked_traces(traces, fs, tlim=None, offset_mode="auto", title="Stacked raw traces", lw=0.45, max_points_per_trace=15_000):
    """Plot stacked traces with display-only downsampling so full sessions render quickly."""
    depths = [-15,-80]
    fig, axes = plt.subplots(1, len(traces), figsize=(14 * len(traces), 10), sharex=True, sharey=False)
    if len(traces) == 1:
        axes = [axes]
    for i,(ax, (dmd, X)) in enumerate(zip(axes, traces.items())):
        n = X.shape[1]
        idx0, idx1 = 0, n
        if tlim is not None:
            idx0 = max(0, int(round(tlim[0] * fs)))
            idx1 = min(n, int(round(tlim[1] * fs)))
        plot_step = max(1, int(np.ceil((idx1 - idx0) / max_points_per_trace)))
        plot_slice = slice(idx0, idx1, plot_step)
        t = np.arange(idx0, idx1, plot_step) / fs
        # Compute dynamic ranges on a downsampled view for speed; plotting is for visual QC, not quantification.
        dynamic_ranges = [np.nanpercentile(x[plot_slice], 99) - np.nanpercentile(x[plot_slice], 1) for x in X]
        offset = 1.2 * np.nanmedian(dynamic_ranges) if offset_mode == "auto" else float(offset_mode)
        offset = max(offset, 1.0)
        yticks = []
        for roi_idx, y in enumerate(X):
            yy = y[plot_slice]
            yy = yy - np.nanmedian(yy)
            ax.plot(t, yy + roi_idx * offset, lw=lw,zorder=-roi_idx)
            yticks.append(roi_idx * offset)
        ax.set_title(f"ASAP8hy VIP somatic voltage (DMD{dmd} = {depths[i]}\u03BCm below pia)")
        ax.set_xlabel("Time (s)")
        ax.set_yticks(yticks)
        ax.set_yticklabels([f"ROI {i}" for i in range(X.shape[0])])
    axes[0].set_ylabel("ROI index")
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig, axes

fig, axes = plot_stacked_traces(traces, FS, title="Full-session raw ASAP8 traces",max_points_per_trace=10800*300)
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"02_full_session_raw_traces"), formats=(".png", ".pdf"))

fig, axes = plot_stacked_traces(traces, FS, tlim=(20, 40), title="First 20 s raw ASAP8 traces")
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"03_first_20s_raw_traces"), formats=(".png", ".pdf"))

## Preprocessing and event detection

The detector has three conceptual steps:

1. Estimate a slow baseline by binning the trace, taking a low percentile within each bin, smoothing across bins, and interpolating back to the sample grid.
2. Convert raw fluorescence into a baseline-subtracted voltage-like signal. Polarity can be auto-inferred, but for this recording depolarizing events are positive-going in raw fluorescence.
3. Detect peaks with `scipy.signal.find_peaks`, then compute widths, post-peak plateau index, decay time, and burst grouping.

Important knobs for fast tuning are `threshold_sigma`, `prominence_sigma`, `baseline_percentile`, and `merge_gap_ms`.

In [ ]:

def estimate_quantile_baseline(trace, fs, bin_s=0.25, percentile=20, smooth_s=2.0):
    """
    Estimate a slow fluorescence baseline without running a huge rolling filter.

    Parameters
    ----------
    trace : array-like, shape (samples,)
    fs : float
        Sample rate in Hz.
    bin_s : float
        Bin duration used before percentile calculation.
    percentile : float
        Low percentile used as a robust baseline anchor within each bin.
    smooth_s : float
        Gaussian smoothing sigma in seconds, applied to the binned baseline.
    """
    y = np.asarray(trace, dtype=np.float32)
    n = y.size
    bin_n = max(1, int(round(bin_s * fs)))
    n_bins = int(np.ceil(n / bin_n))
    vals = np.empty(n_bins, dtype=np.float32)
    centers = np.empty(n_bins, dtype=np.float64)

    for b in range(n_bins):
        i0 = b * bin_n
        i1 = min(n, (b + 1) * bin_n)
        seg = y[i0:i1]
        vals[b] = np.nanpercentile(seg, percentile) if seg.size else np.nan
        centers[b] = 0.5 * (i0 + i1 - 1)

    bad = ~np.isfinite(vals)
    if np.any(bad):
        good = ~bad
        vals[bad] = np.interp(np.flatnonzero(bad), np.flatnonzero(good), vals[good])

    if n_bins > 3 and smooth_s and smooth_s > 0:
        sigma_bins = max(0.5, smooth_s / bin_s)
        vals = ndimage.gaussian_filter1d(vals, sigma=sigma_bins, mode="nearest", truncate=3)

    return np.interp(np.arange(n), centers, vals).astype(np.float32)


def robust_mad(x):
    x = np.asarray(x)
    med = np.nanmedian(x)
    return 1.4826 * np.nanmedian(np.abs(x - med))


def infer_event_polarity(x):
    """Infer whether large events are positive- or negative-going from percentile asymmetry."""
    p1, p50, p99 = np.nanpercentile(x, [1, 50, 99])
    return "positive" if (p99 - p50) >= (p50 - p1) else "negative"


def group_peaks_into_events(peaks, fs, merge_gap_ms=25.0):
    """Group peaks into compound/burst events when neighboring peaks are close in time."""
    peaks = np.asarray(peaks, dtype=int)
    if peaks.size == 0:
        return []
    gap = max(1, int(round(merge_gap_ms / 1000 * fs)))
    groups = []
    current = [int(peaks[0])]
    for p in peaks[1:]:
        p = int(p)
        if p - current[-1] <= gap:
            current.append(p)
        else:
            groups.append(np.asarray(current, dtype=int))
            current = [p]
    groups.append(np.asarray(current, dtype=int))
    return groups


def detect_voltage_events(
    trace,
    fs,
    polarity="auto",
    baseline_bin_s=0.25,
    baseline_percentile=20,
    baseline_smooth_s=2.0,
    smooth_ms=0.25,
    threshold_sigma=4.0,
    prominence_sigma=1.5,
    refractory_ms=3.0,
    min_width_ms=0.10,
    max_width_ms=40.0,
    plateau_window_ms=(8.0, 50.0),
    decay_search_ms=100.0,
    merge_gap_ms=25.0,
):
    """
    Conservative ASAP voltage event detector.

    Returns a dict with detected peaks, peak-level features, event groups, and preprocessed traces.
    A "peak" is an individual local maximum. An "event" can contain one or more close peaks.
    """
    y = np.asarray(trace, dtype=np.float32)
    baseline = estimate_quantile_baseline(
        y, fs,
        bin_s=baseline_bin_s,
        percentile=baseline_percentile,
        smooth_s=baseline_smooth_s,
    )
    x = y - baseline

    if polarity == "auto":
        polarity_used = infer_event_polarity(x)
    else:
        polarity_used = polarity

    # x_det is positive for depolarizing events regardless of optical polarity.
    x_det = -x if polarity_used == "negative" else x

    if smooth_ms and smooth_ms > 0:
        sigma_samples = max(0.1, (smooth_ms / 1000) * fs)
        x_smooth = ndimage.gaussian_filter1d(x_det, sigma=sigma_samples, mode="nearest", truncate=3)
    else:
        x_smooth = x_det

    # For these raw ASAP traces, signal MAD is a practical conservative scale.
    # It intentionally ignores many tiny fluctuations and emphasizes visually obvious events.
    noise = robust_mad(x_smooth)
    if not np.isfinite(noise) or noise <= 0:
        noise = np.nanstd(x_smooth)
    if not np.isfinite(noise) or noise <= 0:
        noise = 1.0

    height = threshold_sigma * noise
    prominence = prominence_sigma * noise
    distance = max(1, int(round(refractory_ms / 1000 * fs)))
    peaks, props = signal.find_peaks(
        x_smooth,
        height=height,
        prominence=prominence,
        distance=distance,
    )

    if peaks.size:
        widths, width_heights, left_ips, right_ips = signal.peak_widths(x_smooth, peaks, rel_height=0.5)
        keep = (widths >= min_width_ms / 1000 * fs) & (widths <= max_width_ms / 1000 * fs)
        peaks = peaks[keep]
        for k in list(props.keys()):
            props[k] = props[k][keep]
        widths = widths[keep]
        width_heights = width_heights[keep]
        left_ips = left_ips[keep]
        right_ips = right_ips[keep]
    else:
        widths = width_heights = left_ips = right_ips = np.array([], dtype=float)

    peak_heights = props.get("peak_heights", np.array([], dtype=float))
    widths_ms = widths / fs * 1000

    # Plateau index: positive sustained depolarization after the peak normalized by peak height.
    post0 = int(round(plateau_window_ms[0] / 1000 * fs))
    post1 = int(round(plateau_window_ms[1] / 1000 * fs))
    plateau_index = np.full(peaks.size, np.nan, dtype=float)
    for i, (p, a) in enumerate(zip(peaks, peak_heights)):
        if p + post1 < x_det.size and a > 0:
            plateau_index[i] = np.nanmean(x_det[p + post0:p + post1]) / a

    # Decay-to-25%-of-peak time. Long/clipped values suggest plateau-like events.
    search_n = int(round(decay_search_ms / 1000 * fs))
    decay25_ms = np.full(peaks.size, np.nan, dtype=float)
    for i, (p, a) in enumerate(zip(peaks, peak_heights)):
        if a <= 0 or p >= x_smooth.size - 1:
            continue
        tail = x_smooth[p:min(x_smooth.size, p + search_n)]
        below = np.flatnonzero(tail <= 0.25 * a)
        if below.size:
            decay25_ms[i] = below[0] / fs * 1000
        else:
            decay25_ms[i] = decay_search_ms

    groups = group_peaks_into_events(peaks, fs=fs, merge_gap_ms=merge_gap_ms)
    peak_to_event = np.full(peaks.size, -1, dtype=int)
    peak_is_compound = np.zeros(peaks.size, dtype=bool)
    p_to_i = {int(p): i for i, p in enumerate(peaks)}
    for event_idx, group in enumerate(groups):
        for p in group:
            i = p_to_i.get(int(p))
            if i is not None:
                peak_to_event[i] = event_idx
                peak_is_compound[i] = group.size > 1

    return {
        "peaks": peaks,
        "peak_times_s": peaks / fs,
        "props": props,
        "baseline": baseline,
        "x": x,
        "x_det": x_det,
        "x_smooth": x_smooth,
        "noise": float(noise),
        "height_threshold": float(height),
        "prominence_threshold": float(prominence),
        "polarity": polarity_used,
        "widths_ms": widths_ms,
        "left_ips": left_ips,
        "right_ips": right_ips,
        "peak_heights_raw_units": peak_heights,
        "peak_heights_dff": peak_heights / np.maximum(baseline[peaks], 1e-9) if peaks.size else np.array([]),
        "plateau_index": plateau_index,
        "decay25_ms": decay25_ms,
        "events": groups,
        "peak_to_event": peak_to_event,
        "peak_is_compound": peak_is_compound,
        "params": {
            "baseline_bin_s": baseline_bin_s,
            "baseline_percentile": baseline_percentile,
            "baseline_smooth_s": baseline_smooth_s,
            "smooth_ms": smooth_ms,
            "threshold_sigma": threshold_sigma,
            "prominence_sigma": prominence_sigma,
            "refractory_ms": refractory_ms,
            "min_width_ms": min_width_ms,
            "max_width_ms": max_width_ms,
            "plateau_window_ms": plateau_window_ms,
            "decay_search_ms": decay_search_ms,
            "merge_gap_ms": merge_gap_ms,
        },
    }

## Run event detection for each ROI

Start conservative. For a slide-ready first pass, it is better to miss small events than to count noise as spikes. Lower `threshold_sigma` if the overlay plot below clearly misses real single spikes.

In [ ]:

SPIKE_PARAMS = dict(
    polarity="auto",             # use "positive" for this recording if you want to lock it down
    baseline_bin_s=0.25,
    baseline_percentile=20,
    baseline_smooth_s=2.0,
    smooth_ms=0.25,
    threshold_sigma=4.0,
    prominence_sigma=1.5,
    refractory_ms=3.0,
    min_width_ms=0.10,
    max_width_ms=40.0,
    plateau_window_ms=(8.0, 50.0),
    decay_search_ms=100.0,
    merge_gap_ms=25.0,
)

results = {}
summary_rows = []

for dmd, X in traces.items():
    results[dmd] = []
    duration_s = X.shape[1] / FS
    for roi_idx, y in enumerate(X):
        res = detect_voltage_events(y, FS, **SPIKE_PARAMS)
        results[dmd].append(res)
        n_peaks = len(res["peaks"])
        n_events = len(res["events"])
        n_compound_events = sum(len(g) > 1 for g in res["events"])
        n_peaks_in_compound = int(np.sum(res["peak_is_compound"])) if n_peaks else 0
        baseline = res["baseline"]
        drift_pct = 100 * (np.nanpercentile(baseline, 95) - np.nanpercentile(baseline, 5)) / np.nanmedian(baseline)
        summary_rows.append({
            "dmd": dmd,
            "roi": roi_idx,
            "label": f"DMD{dmd}_ROI{roi_idx}",
            "duration_s": duration_s,
            "n_peaks": n_peaks,
            "peak_rate_hz": n_peaks / duration_s,
            "n_events": n_events,
            "event_rate_hz": n_events / duration_s,
            "polarity": res["polarity"],
            "noise_raw_units": res["noise"],
            "median_peak_amp_raw": np.nanmedian(res["peak_heights_raw_units"]) if n_peaks else np.nan,
            "median_peak_amp_dff_pct": 100 * np.nanmedian(res["peak_heights_dff"]) if n_peaks else np.nan,
            "median_width_ms": np.nanmedian(res["widths_ms"]) if n_peaks else np.nan,
            "median_decay25_ms": np.nanmedian(res["decay25_ms"]) if n_peaks else np.nan,
            "median_plateau_index": np.nanmedian(res["plateau_index"]) if n_peaks else np.nan,
            "p90_plateau_index": np.nanpercentile(res["plateau_index"], 90) if n_peaks else np.nan,
            "compound_event_fraction": n_compound_events / n_events if n_events else np.nan,
            "peaks_in_compound_fraction": n_peaks_in_compound / n_peaks if n_peaks else np.nan,
            "baseline_drift_5_95_pct": drift_pct,
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(3))

if SAVE_FIGURES:
    Path(FIG_DIR).mkdir(exist_ok=True, parents=True)
    summary_df.to_csv(os.path.join(FIG_DIR,"asap8_roi_voltage_summary.csv"), index=False)

## Detection overlays

These panels are the main QC check. Red ticks are detected peaks. The dotted horizontal line is the ROI-specific detection threshold. If the markers do not match your visual intuition, tune `SPIKE_PARAMS` and rerun detection.

In [ ]:

def plot_detection_overlay(traces, results, fs, tlim=(0, 20), use_detrended=True, max_points_per_trace=15_000):
    """Overlay detected peaks on a zoomed trace window; downsample only the line display."""
    n_rows = sum(X.shape[0] for X in traces.values())
    fig, axes = plt.subplots(n_rows, 1, figsize=(14, max(2.0 * n_rows, 4)), sharex=True)
    if n_rows == 1:
        axes = [axes]
    row = 0
    for dmd, X in traces.items():
        idx0 = max(0, int(round(tlim[0] * fs)))
        idx1 = min(X.shape[1], int(round(tlim[1] * fs)))
        plot_step = max(1, int(np.ceil((idx1 - idx0) / max_points_per_trace)))
        plot_idx = np.arange(idx0, idx1, plot_step)
        t = plot_idx / fs
        for roi_idx, y in enumerate(X):
            ax = axes[row]
            res = results[dmd][roi_idx]
            yy = res["x_det"][plot_idx] if use_detrended else y[plot_idx]
            ax.plot(t, yy, lw=0.55)
            in_win = (res["peaks"] >= idx0) & (res["peaks"] < idx1)
            p = res["peaks"][in_win]
            if p.size:
                y_peak = res["x_det"][p] if use_detrended else y[p]
                ax.plot(p / fs, y_peak, "v", ms=4)
            if use_detrended:
                ax.axhline(res["height_threshold"], ls="--", lw=0.8)
                ax.set_ylabel("F - F0")
            else:
                ax.set_ylabel("Raw F")
            ax.set_title(f"DMD{dmd} ROI{roi_idx}: {p.size} detected peaks in {tlim[0]}-{tlim[1]} s")
            row += 1
    axes[-1].set_xlabel("Time (s)")
    fig.tight_layout()
    return fig, axes

fig, axes = plot_detection_overlay(traces, results, FS, tlim=(0, 10), use_detrended=True)
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"04_detection_overlay_first_10s"), formats=(".png", ".pdf"))

# A second zoom window around a visibly active period can help show compound/plateau dynamics.
fig, axes = plot_detection_overlay(traces, results, FS, tlim=(160, 170), use_detrended=True)
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"05_detection_overlay_160_170s"), formats=(".png", ".pdf"))

## Spike-triggered/event-triggered averages by ROI

These are triggered on detected peak times. Individual snippets are plotted faintly; the bold line is the mean and the shaded band is SEM. For compound ROIs, the average can broaden substantially or retain a sustained post-peak component.

In [ ]:

def extract_event_windows(signal_1d, peaks, fs, window_ms=(-20, 80), max_events=250, seed=0):
    signal_1d = np.asarray(signal_1d)
    peaks = np.asarray(peaks, dtype=int)
    pre = int(round(abs(window_ms[0]) / 1000 * fs))
    post = int(round(window_ms[1] / 1000 * fs))
    valid = peaks[(peaks - pre >= 0) & (peaks + post < signal_1d.size)]
    if valid.size > max_events:
        rng = np.random.default_rng(seed)
        valid = np.sort(rng.choice(valid, size=max_events, replace=False))
    snippets = np.stack([signal_1d[p - pre:p + post + 1] for p in valid], axis=0) if valid.size else np.empty((0, pre + post + 1))
    t_ms = (np.arange(-pre, post + 1) / fs) * 1000
    return t_ms, snippets, valid


def plot_spike_triggered_averages(results, fs, window_ms=(-20, 100), max_events=250, normalize=False):
    n_rows = sum(len(v) for v in results.values())
    fig, axes = plt.subplots(n_rows, 1, figsize=(9.5, max(2.0 * n_rows, 4)), sharex=True)
    if n_rows == 1:
        axes = [axes]
    row = 0
    for dmd, dmd_results in results.items():
        for roi_idx, res in enumerate(dmd_results):
            ax = axes[row]
            t_ms, snippets, used_peaks = extract_event_windows(
                res["x_det"], res["peaks"], fs, window_ms=window_ms, max_events=max_events, seed=roi_idx + 100 * dmd
            )
            if snippets.size:
                if normalize:
                    denom = np.nanmax(snippets, axis=1, keepdims=True)
                    denom[denom == 0] = np.nan
                    snippets_to_plot = snippets / denom
                    ylabel = "Norm. amplitude"
                else:
                    snippets_to_plot = snippets
                    ylabel = "F - F0"
                mean = np.nanmean(snippets_to_plot, axis=0)
                sem = np.nanstd(snippets_to_plot, axis=0) / np.sqrt(snippets_to_plot.shape[0])
                # Individual event snippets are helpful for showing heterogeneity.
                for s in snippets_to_plot[:min(80, snippets_to_plot.shape[0])]:
                    ax.plot(t_ms, s, lw=0.35, alpha=0.08)
                ax.plot(t_ms, mean, lw=2.0)
                ax.fill_between(t_ms, mean - sem, mean + sem, alpha=0.25, linewidth=0)
                ax.axvline(0, lw=0.8, ls="--")
                ax.axhline(0, lw=0.5)
                ax.set_title(f"DMD{dmd} ROI{roi_idx}: STA/ETA, n={snippets_to_plot.shape[0]} sampled peaks")
                ax.set_ylabel(ylabel)
            else:
                ax.text(0.5, 0.5, "No valid detected events", transform=ax.transAxes, ha="center", va="center")
                ax.set_title(f"DMD{dmd} ROI{roi_idx}")
            row += 1
    axes[-1].set_xlabel("Time from detected peak (ms)")
    fig.tight_layout()
    return fig, axes

fig, axes = plot_spike_triggered_averages(results, FS, window_ms=(-20, 100), max_events=250, normalize=False)
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"06_spike_triggered_averages_raw_units"), formats=(".png", ".pdf"))

fig, axes = plot_spike_triggered_averages(results, FS, window_ms=(-20, 100), max_events=250, normalize=True)
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"07_spike_triggered_averages_normalized"), formats=(".png", ".pdf"))

## ROI-level feature summaries

These summaries are meant to quickly identify ROIs with different voltage phenotypes:

- **Peak/event rate**: how often the detector finds candidate events.
- **Median width and decay time**: broader/longer events suggest compound events or plateau-like depolarizations.
- **Plateau index**: post-peak signal from 8–50 ms divided by peak height; larger values mean more sustained post-peak depolarization.
- **Compound fraction**: fraction of detected events with more than one close peak.

In [ ]:

def barplot_summary(summary_df, columns, title=None):
    labels = summary_df["label"].to_list()
    fig, axes = plt.subplots(len(columns), 1, figsize=(10, 2.2 * len(columns)), sharex=True)
    if len(columns) == 1:
        axes = [axes]
    x = np.arange(len(labels))
    for ax, col in zip(axes, columns):
        ax.bar(x, summary_df[col].values)
        ax.set_ylabel(col.replace("_", "\n"))
        ax.grid(axis="y", alpha=0.25)
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(labels, rotation=35, ha="right")
    if title:
        fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig, axes

feature_cols = [
    "peak_rate_hz",
    "median_peak_amp_dff_pct",
    "median_width_ms",
    "median_decay25_ms",
    "median_plateau_index",
    "compound_event_fraction",
    "baseline_drift_5_95_pct",
]
fig, axes = barplot_summary(summary_df, feature_cols, title="Per-ROI voltage/event features")
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"08_per_roi_feature_bars"), formats=(".png", ".pdf"))

# Scatter: a compact phenotype view.
fig, ax = plt.subplots(figsize=(6.5, 5.2))
size = 80 + 30 * np.nan_to_num(summary_df["median_peak_amp_dff_pct"].values, nan=0)
ax.scatter(summary_df["peak_rate_hz"], summary_df["median_plateau_index"], s=size, alpha=0.75)
for _, row in summary_df.iterrows():
    ax.text(row["peak_rate_hz"], row["median_plateau_index"], row["label"], fontsize=8, ha="left", va="bottom")
ax.set_xlabel("Detected peak rate (Hz)")
ax.set_ylabel("Median plateau index")
ax.set_title("ROI phenotype: event rate vs sustained post-peak depolarization")
ax.grid(alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"09_roi_phenotype_scatter"), formats=(".png", ".pdf"))

## Single-like versus compound/plateau event examples

This section pulls out example snippets based on plateau index and compound grouping. It is useful for slides because it shows why a single threshold/count is not enough: some ROIs have crisp isolated events, while others have clustered peaks and sustained depolarization.

In [ ]:

def plot_event_examples(results, fs, max_examples=8, window_ms=(-30, 150)):
    n_rows = sum(len(v) for v in results.values())
    fig, axes = plt.subplots(n_rows, 2, figsize=(12, max(2.2 * n_rows, 4)), sharex=True)
    if n_rows == 1:
        axes = np.asarray([axes])

    row = 0
    for dmd, dmd_results in results.items():
        for roi_idx, res in enumerate(dmd_results):
            peaks = res["peaks"]
            plateau = res["plateau_index"]
            compound = res["peak_is_compound"]
            labels = ["low plateau / isolated", "high plateau or compound"]
            selections = []
            if peaks.size:
                isolated_low = np.flatnonzero((~compound) & np.isfinite(plateau))
                if isolated_low.size:
                    isolated_low = isolated_low[np.argsort(plateau[isolated_low])[:max_examples]]
                selections.append(isolated_low)

                high_or_compound = np.flatnonzero((compound) | (plateau >= np.nanpercentile(plateau, 80)))
                if high_or_compound.size:
                    order = np.argsort(np.nan_to_num(plateau[high_or_compound], nan=-np.inf))[::-1]
                    high_or_compound = high_or_compound[order[:max_examples]]
                selections.append(high_or_compound)
            else:
                selections = [[], []]

            for col, sel in enumerate(selections):
                ax = axes[row, col]
                if len(sel):
                    selected_peaks = peaks[np.asarray(sel, dtype=int)]
                    t_ms, snippets, _ = extract_event_windows(res["x_det"], selected_peaks, fs, window_ms=window_ms, max_events=max_examples)
                    for s in snippets:
                        ax.plot(t_ms, s, lw=1.0, alpha=0.75)
                    ax.axvline(0, lw=0.8, ls="--")
                    ax.axhline(0, lw=0.5)
                else:
                    ax.text(0.5, 0.5, "No examples", transform=ax.transAxes, ha="center", va="center")
                ax.set_title(f"DMD{dmd} ROI{roi_idx}: {labels[col]}")
                ax.set_ylabel("F - F0")
            row += 1
    for ax in axes[-1, :]:
        ax.set_xlabel("Time from selected peak (ms)")
    fig.tight_layout()
    return fig, axes

fig, axes = plot_event_examples(results, FS, max_examples=8, window_ms=(-30, 150))
if SAVE_FIGURES:
    save_figure(fig, os.path.join(FIG_DIR,"10_single_vs_compound_event_examples"), formats=(".png", ".pdf"))

## Between-ROI comparisons: event-rate correlations and coincidence

These are intentionally simple, but useful for asking whether ROIs share event timing. Correlations are computed from binned detected-peak counts, while coincidence is the fraction of peaks in one ROI that have at least one peak in the other ROI within a short tolerance.

In [ ]:

def flatten_results(results):
    flat = []
    for dmd, dmd_results in results.items():
        for roi_idx, res in enumerate(dmd_results):
            flat.append({"dmd": dmd, "roi": roi_idx, "label": f"DMD{dmd}_ROI{roi_idx}", "result": res})
    return flat


def binned_peak_counts(peaks, n_samples, fs, bin_s=0.05):
    n_bins = int(np.ceil(n_samples / (bin_s * fs)))
    edges = np.arange(n_bins + 1) * bin_s * fs
    counts, _ = np.histogram(peaks, bins=edges)
    return counts.astype(float) / bin_s


def directional_coincidence_fraction(peaks_a, peaks_b, fs, tolerance_ms=10.0):
    peaks_a = np.asarray(peaks_a, dtype=int)
    peaks_b = np.asarray(peaks_b, dtype=int)
    if peaks_a.size == 0 or peaks_b.size == 0:
        return np.nan
    tol = int(round(tolerance_ms / 1000 * fs))
    peaks_b = np.sort(peaks_b)
    hits = 0
    for p in peaks_a:
        j = np.searchsorted(peaks_b, p)
        near = False
        if j < peaks_b.size and abs(peaks_b[j] - p) <= tol:
            near = True
        if j > 0 and abs(peaks_b[j - 1] - p) <= tol:
            near = True
        hits += int(near)
    return hits / peaks_a.size

flat = flatten_results(results)
labels = [r["label"] for r in flat]
min_samples = min(traces[dmd].shape[1] for dmd in traces)
BIN_S = 0.05
rate_mat = np.vstack([binned_peak_counts(r["result"]["peaks"], min_samples, FS, bin_s=BIN_S) for r in flat])
rate_corr = np.corrcoef(rate_mat)

coinc = np.full((len(flat), len(flat)), np.nan)
for i, ri in enumerate(flat):
    for j, rj in enumerate(flat):
        if i == j:
            coinc[i, j] = 1.0
        else:
            cij = directional_coincidence_fraction(ri["result"]["peaks"], rj["result"]["peaks"], FS, tolerance_ms=10.0)
            cji = directional_coincidence_fraction(rj["result"]["peaks"], ri["result"]["peaks"], FS, tolerance_ms=10.0)
            coinc[i, j] = np.nanmean([cij, cji])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
for ax, mat, title, vmin, vmax in [
    (axes[0], rate_corr, f"Peak-rate correlation\n{BIN_S*1000:.0f} ms bins", -1, 1),
    (axes[1], coinc, "Symmetric peak coincidence\n±10 ms", 0, 1),
]:
    im = ax.imshow(mat, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
# if SAVE_FIGURES:
#     save_figure(fig, FIG_DIR / "11_between_roi_timing", formats=(".png", ".pdf"))

## Slow baseline / plateau burden and spectral summaries

This is a quick way to see which ROIs carry slow fluorescence changes versus sharp events. For more rigorous analysis, this should eventually be split into explicit voltage bands after validating sample timing and filtering choices.

In [ ]:

def welch_bandpowers(x, fs, bands=((0.2, 5), (5, 30), (30, 200), (200, 1000)), nperseg_s=4.0):
    nperseg = int(round(nperseg_s * fs))
    nperseg = min(max(nperseg, 256), x.size)
    freqs, psd = signal.welch(x, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
    out = {}
    for lo, hi in bands:
        sel = (freqs >= lo) & (freqs < hi)
        out[f"power_{lo:g}_{hi:g}_Hz"] = np.trapz(psd[sel], freqs[sel]) if np.any(sel) else np.nan
    return freqs, psd, out

spectral_rows = []
for dmd, X in traces.items():
    for roi_idx, y in enumerate(X):
        res = results[dmd][roi_idx]
        # Use the detrended, polarity-corrected signal so positive deflections are depolarizing.
        freqs, psd, bands = welch_bandpowers(res["x_det"], FS)
        row = {"dmd": dmd, "roi": roi_idx, "label": f"DMD{dmd}_ROI{roi_idx}"}
        row.update(bands)
        spectral_rows.append(row)

spectral_df = pd.DataFrame(spectral_rows)
display(spectral_df.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
for dmd, dmd_results in results.items():
    for roi_idx, res in enumerate(dmd_results):
        freqs, psd, _ = welch_bandpowers(res["x_det"], FS)
        ax.loglog(freqs[1:], psd[1:], lw=1.0, label=f"DMD{dmd} ROI{roi_idx}")
ax.set_xlim(0.2, 1000)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD of F-F0")
ax.set_title("Per-ROI voltage trace spectra")
ax.legend(ncol=2, frameon=False)
fig.tight_layout()
# if SAVE_FIGURES:
#     save_figure(fig, FIG_DIR / "12_roi_trace_spectra", formats=(".png", ".pdf"))

# Slow baseline drift over the session, normalized to each ROI median baseline.
fig, axes = plt.subplots(len(traces), 1, figsize=(11, 2.5 * len(traces)), sharex=True)
if len(traces) == 1:
    axes = [axes]
for ax, (dmd, X) in zip(axes, traces.items()):
    for roi_idx, res in enumerate(results[dmd]):
        baseline = res["baseline"]
        # Downsample for plotting only.
        step = max(1, int(round(FS * 0.25)))
        t_ds = np.arange(0, baseline.size, step) / FS
        b_ds = baseline[::step]
        b_pct = 100 * (b_ds / np.nanmedian(baseline) - 1)
        ax.plot(t_ds, b_pct, lw=1.0, label=f"ROI{roi_idx}")
    ax.set_title(f"DMD{dmd}: slow baseline drift")
    ax.set_ylabel("Baseline Δ%")
    ax.legend(frameon=False, ncol=min(4, X.shape[0]))
axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
# if SAVE_FIGURES:
#     save_figure(fig, FIG_DIR / "13_slow_baseline_drift", formats=(".png", ".pdf"))

## Notes for interpretation / next steps

- The current detector is good for a **fast meeting view**, but it should not yet be treated as a validated spike sorter.
- ROIs with high plateau index, long decay, and high compound fraction likely need a separate event model: one component for fast peaks and another for sustained depolarizing plateaus.
- For a cleaner next pass, consider producing both: (1) fast spike calls from a high-pass or deconvolution-like trace, and (2) plateau/burst epochs from a low-frequency or envelope trace.
- The dF/F amplitude columns are useful for between-ROI comparisons, but raw amplitude and noise should also be checked because expression/brightness vary strongly across ROIs and DMDs.

In [ ]:
from vip_slap2_analysis.voltage.summary import VoltageSummary

In [ ]:
vs = VoltageSummary(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-10_13-02-11\826032_2026-02-10_13-02-11_slap2_2026-02-10_13-02-11\source_extraction\dendriticVoltageExtraction\dendriticVoltageSummary-260624-081356.mat")

In [ ]:
vs.metadata